# LinMulT Quick Start

Minimal examples for `LinT` (single-modality) and `LinMulT` (multimodal) covering common patterns:
sequence output, time-aggregation, masking, multiple heads, missing modalities, and TAM fusion.

Install: `pip install linmult`

In [ ]:
import torch

from linmult import LinMulT, LinMulTConfig, LinT, LinTConfig, apply_logit_aggregation

B = 4  # batch size
print(f"torch {torch.__version__}")

## LinT — single-modality transformer

In [ ]:
# Sequence output: (B, T, output_dim)
x = torch.rand(B, 300, 25)
model = LinT(
    LinTConfig(
        input_feature_dim=25,
        heads=[{"name": "out", "type": "simple", "output_dim": 5}],
    )
)
result = model(x)
print(f"input:  {tuple(x.shape)}")
print(f"output: {tuple(result['out'].shape)}")  # (B, T, 5)

In [ ]:
# Aggregated output via time_dim_reducer: (B, output_dim)
model = LinT(
    LinTConfig(
        input_feature_dim=25,
        heads=[{"name": "out", "type": "simple", "output_dim": 5}],
        time_dim_reducer="attentionpool",
    )
)
result = model(x)
print(f"input:  {tuple(x.shape)}")
print(f"output: {tuple(result['out'].shape)}")  # (B, 5)

In [ ]:
# With mask and multiple heads
mask = (torch.arange(x.size(1)).unsqueeze(0) < x.size(1) - 50).expand(B, -1)
model = LinT(
    LinTConfig(
        input_feature_dim=25,
        heads=[
            {"name": "seq", "type": "sequence", "output_dim": 5},
            {"name": "agg", "type": "sequence_aggregation", "output_dim": 3},
        ],
    )
)
result = model(x, mask)
print(f"input:  {tuple(x.shape)}  mask: {tuple(mask.shape)}")
print(f"seq:    {tuple(result['seq'].shape)}")  # (B, T, 5)
print(f"agg:    {tuple(result['agg'].shape)}")  # (B, 3)

## LinMulT — multimodal transformer

In [ ]:
# 2 modalities, aggregated output
x1 = torch.rand(B, 300, 25)
x2 = torch.rand(B, 100, 35)
model = LinMulT(
    LinMulTConfig(
        input_feature_dim=[25, 35],
        heads=[{"name": "out", "type": "simple", "output_dim": 5}],
        time_dim_reducer="gap",
    )
)
result = model([x1, x2])
print(f"x1: {tuple(x1.shape)}  x2: {tuple(x2.shape)}")
print(f"output: {tuple(result['out'].shape)}")  # (B, 5)

In [ ]:
# 3 modalities — third modality fully absent, TAM fusion
x3 = torch.rand(B, 100, 64)
mask1 = (torch.arange(x1.size(1)).unsqueeze(0) < x1.size(1) - 30).expand(B, -1)
mask2 = (torch.arange(x2.size(1)).unsqueeze(0) < x2.size(1) - 10).expand(B, -1)
mask3_absent = torch.zeros(B, x3.size(1), dtype=torch.bool)  # fully absent modality

model = LinMulT(
    LinMulTConfig(
        input_feature_dim=[25, 35, 64],
        heads=[
            {"name": "seq", "type": "sequence", "output_dim": 5},
            {"name": "agg", "type": "sequence_aggregation", "output_dim": 3},
        ],
        add_module_multimodal_signal=True,
        tam_aligner="amp",
        add_module_tam_fusion=True,
        tam_time_dim=100,
    )
)
result = model([x1, x2, x3], [mask1, mask2, mask3_absent])
print(f"x1: {tuple(x1.shape)}  x2: {tuple(x2.shape)}  x3: absent (zero-mask)")
print(f"seq: {tuple(result['seq'].shape)}")  # (B, 100, 5)
print(f"agg: {tuple(result['agg'].shape)}")  # (B, 3)

## apply_logit_aggregation

Aggregate frame-level logits to a clip-level prediction.

In [7]:
logits = torch.rand(B, 300, 5)  # (batch, time, classes)
for method in ("meanpooling", "maxpooling"):
    agg = apply_logit_aggregation(logits, method=method)
    print(f"{method:20s}: {tuple(agg.shape)}")  # (B, 5)

meanpooling         : (4, 5)
maxpooling          : (4, 5)
